# 09 · Uncertainty and Resolution

A slip map is only half the answer. The other half is: *how well is it actually
resolved?* Which patches does the data constrain, and which are just smoothing
filling in blanks? This notebook quantifies that with the **posterior
covariance**, the **resolution matrix**, and a **checkerboard test**, and
reports the moment magnitude with an uncertainty.

## Learning objectives
- Compute per-patch slip uncertainty from the posterior covariance.
- Read the model **resolution matrix** and its diagonal.
- Run a **checkerboard test** to map where slip is recoverable.
- Report moment magnitude $M_w$ with an error bar.

## Posterior covariance and resolution

For the linear-Gaussian problem, the posterior **model covariance** is

$$ C_m = \left(G^{\mathsf T} W G + \lambda L^{\mathsf T} L\right)^{-1}. $$

Its diagonal gives the per-patch variance; $\sqrt{\operatorname{diag} C_m}$ is
the 1-$\sigma$ slip uncertainty.

The **model resolution matrix** relates the recovered model to the truth,

$$ \hat{\mathbf m} = R\,\mathbf m_{\text{true}}, \qquad R = G^{-g} G, $$

where $G^{-g}$ is the generalized inverse. Each recovered patch is a *weighted
average* of the true slip — the corresponding **row of $R$** is that patch's
resolution kernel. If $R \approx I$, slip is perfectly resolved; broad rows mean
the estimate is a blurred average of its neighbours.

There is an unavoidable **trade-off**: strong smoothing lowers uncertainty but
smears resolution; weak smoothing sharpens resolution but inflates uncertainty.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.greens(
    fault, geodef.GNSS(lon=glon, lat=glat, ve=_zero, vn=_zero, vu=_zero, se=_one, sn=_one, su=_one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.GNSS(
    lon=glon, lat=glat,
    ve=d_obs[0::3], vn=d_obs[1::3], vu=d_obs[2::3],
    se=np.full(n_sta, sigma), sn=np.full(n_sta, sigma), su=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

40 patches, 36 stations, 108 observations


In [2]:
result = geodef.invert(fault, gnss, components="dip",
                       smoothing="laplacian", smoothing_strength=1.0)

## Per-patch uncertainty

`model_uncertainty` returns the 1-$\sigma$ slip error for each patch. Deep,
poorly-sampled patches carry the largest uncertainty.

In [3]:
unc = geodef.model_uncertainty(result, fault, gnss)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
geodef.plot.slip(fault, result.slip_vector, ax=ax1, title="Recovered slip",
                 colorbar_label="Slip (m)")
geodef.plot.uncertainty(fault, unc, ax=ax2, title="1-sigma uncertainty")
plt.tight_layout()

## Resolution

`model_resolution` returns the diagonal of $R$. Values near 1 mark
well-resolved patches (typically shallow, beneath the station network); values
near 0 mark patches the data barely see.

In [4]:
R = geodef.model_resolution(result, fault, gnss)   # full (N, N) matrix
res_diag = np.diag(R)                              # per-patch resolution
geodef.plot.resolution(fault, res_diag, vmin=0, vmax=1,
                       title="Model resolution (diagonal of R)")
print(f"resolution range: {res_diag.min():.2f} - {res_diag.max():.2f}")

resolution range: -0.28 - 0.90


## Checkerboard test

A classic recoverability check: put a *known* checkerboard of slip on the fault,
generate synthetic data, invert, and see where the checkers survive. Sharp
checkers = well resolved; smeared or missing checkers = poorly resolved.

In [5]:
checker = 1.0 + 0.9 * ((-1.0) ** i) * ((-1.0) ** j)     # alternating high/low
m_check = np.concatenate([np.zeros(N), checker])
d_check = G_full @ m_check + rng.normal(0.0, sigma, G_full.shape[0])
gnss_check = geodef.GNSS(lon=glon, lat=glat, ve=d_check[0::3], vn=d_check[1::3], vu=d_check[2::3],
                         se=np.full(n_sta, sigma), sn=np.full(n_sta, sigma), su=np.full(n_sta, sigma))
rec = geodef.invert(fault, gnss_check, components="dip",
                    smoothing="laplacian", smoothing_strength=0.3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
geodef.plot.slip(fault, checker, ax=ax1, title="Input checkerboard",
                 colorbar_label="Slip (m)")
geodef.plot.slip(fault, rec.slip_vector, ax=ax2, title="Recovered",
                 colorbar_label="Slip (m)")
plt.tight_layout()

## Moment magnitude with uncertainty

The scalar seismic moment $M_0 = \mu \sum_k s_k A_k$ is linear in slip, so its
uncertainty propagates from the slip covariance. Here we approximate it by
sampling slip from the posterior.

In [6]:
Cm = geodef.model_covariance(result, fault, gnss)
samples = rng.multivariate_normal(result.slip_vector, Cm, size=500)
# each sample is a length-N per-patch slip; magnitude() takes per-patch slip.
mw = np.array([fault.magnitude(np.clip(s, 0, None)) for s in samples])
print(f"Mw = {mw.mean():.2f} +/- {mw.std():.2f}")

Mw = 7.07 +/- 0.01


## Exercises
1. **Trade-off.** Recompute uncertainty and resolution at `smoothing_strength=0.1`
   and `10`. Confirm that stronger smoothing lowers uncertainty but blurs resolution.
2. **Checker size.** Double the checkerboard wavelength (checker every *two*
   patches). Are the larger checkers better recovered? Relate this to the
   resolution map.
3. **Report.** Quote your preferred model's $M_w \pm$ uncertainty and the fraction
   of patches with resolution above 0.5.

## Checkpoint questions
1. What does a broad row of the resolution matrix tell you about a patch?
2. Why do deeper patches have larger uncertainty?
3. Why does a checkerboard test reveal resolution that a single slip model cannot?

## Summary
- $C_m = (G^{\mathsf T} W G + \lambda L^{\mathsf T} L)^{-1}$ gives per-patch uncertainty.
- The resolution matrix $R = G^{-g}G$ shows where slip is truly constrained.
- Checkerboard tests map recoverability; moment uncertainty follows from $C_m$.

**Next:** notebook 10 relaxes the fixed geometry and searches for the fault
parameters themselves.